# 05 Deskew and HOG

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`EXP011_KNN_Normal_Deskew.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/scripts/EXP011_KNN_Normal_Deskew.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/logs)
- [`EXP012_KNN_Normal_HOG.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/scripts/EXP012_KNN_Normal_HOG.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/logs)
- [`EXP013_KNN_Normal_HOG_Mix_Ratio.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/scripts/EXP013_KNN_Normal_HOG_Mix_Ratio.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/logs)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'05_model_optimize/KNN/Normal/scripts/EXP011_KNN_Normal_Deskew.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'a71e6d32d0662e65d79182c98d17274a44fbf397'}, '05_model_optimize/KNN/Normal/scripts/EXP012_KNN_Normal_HOG.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'bc8ef6102d72ba49e0d9f4dc49ca66cb7adec6b4'}, '05_model_optimize/KNN/Normal/scripts/EXP013_KNN_Normal_HOG_Mix_Ratio.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '6262bddbd2c8a459bad6ee2d9bca3b64edc2a103'}}

## EXP011_KNN_Normal_Deskew

원본: [05_model_optimize/KNN/Normal/scripts/EXP011_KNN_Normal_Deskew.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/scripts/EXP011_KNN_Normal_Deskew.py)  
Git blob: `a71e6d32d0662e65d79182c98d17274a44fbf397`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/logs)

EXP-011: image-moment deskew for KNN Normal, retaining EXP-009 augmentation.

Run from repository root:
  python 05_model_optimize/KNN/Normal/scripts/EXP011_KNN_Normal_Deskew.py

Required scripts in the same folder:
  EXP004_KNN_Normal_CoM.py
  EXP006_KNN_Normal_BB_Size_fixed.py
  EXP007_KNN_Normal_Augmentation.py

Compare:
  com_control: CoM (EXP-009 best, no deskew)
  deskew_then_com: image-moment deskew, then CoM
  com_then_deskew: CoM, then image-moment deskew
All variants share the exact EX



In [ ]:
%%archive_source 05_model_optimize/KNN/Normal/scripts/EXP011_KNN_Normal_Deskew.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""EXP-011: image-moment deskew for KNN Normal, retaining EXP-009 augmentation.

Run from repository root:
  python 05_model_optimize/KNN/Normal/scripts/EXP011_KNN_Normal_Deskew.py

Required scripts in the same folder:
  EXP004_KNN_Normal_CoM.py
  EXP006_KNN_Normal_BB_Size_fixed.py
  EXP007_KNN_Normal_Augmentation.py

Compare:
  com_control: CoM (EXP-009 best, no deskew)
  deskew_then_com: image-moment deskew, then CoM
  com_then_deskew: CoM, then image-moment deskew
All variants share the exact EXP-009 30% combined augmentation, stroke=0.44,
KNN k=3 distance Euclidean, official training only. No test-set leakage.
"""
from __future__ import annotations

import json
import platform
import sys
import time
from pathlib import Path

import numpy as np
import sklearn
from scipy.ndimage import map_coordinates

ROOT = Path(__file__).resolve().parents[4]
SCRIPTS = ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'scripts'
sys.path.insert(0, str(ROOT / '03_experiments' / 'model_optimization'))
sys.path.insert(0, str(SCRIPTS))

from common import fixed_split, LABELS, scores
from EXP004_KNN_Normal_CoM import BatchedCoM, read_handmade, paired_stats
from EXP007_KNN_Normal_Augmentation import (
    knn_model, sample_balanced, size_jitter, stroke_jitter,
    test_augmentation, save_csv, save_cm, OFFICIAL_GUARDRAIL, SEED,
)

OUTPUT = ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'results' / 'EXP011_deskew'
AUG_FRACTION = 0.30
STROKE_STRENGTH = 0.44
EXPECTED_CONTROL_CORRECT = (18381, 2153)
THRESHOLD = 32.0
MAX_SLOPE = 0.45
PIXEL_GRID_Y, PIXEL_GRID_X = np.indices((28, 28), dtype=np.float64)


def deskew(batch, batch_size=512):
    """Label-free moment shear correction, bilinear interpolation, constant-zero border.

    slope = mu11 / mu02 where mu11=cov(x,y) and mu02=var(y).
    For output (x,y), sample input (x + slope*(y-cy), y).
    Near-empty or y-degenerate glyphs remain unchanged. Clamp slope for safety.
    Input/output pixel range: 0..255. Input shape: (N,784).
    """
    src = np.asarray(batch, dtype=np.float32)
    if src.ndim != 2 or src.shape[1] != 784:
        raise ValueError(f'Expected (N,784), got {src.shape}')
    out = np.empty_like(src)
    clipped = skipped = changed = 0
    for i in range(len(src)):
        img = src[i].reshape(28, 28)
        mass = np.where(img >= THRESHOLD, img, 0).astype(np.float64)
        total = mass.sum()
        if total <= 1e-9:
            out[i] = src[i]
            skipped += 1
            continue
        cx = float(np.sum(mass * PIXEL_GRID_X) / total)
        cy = float(np.sum(mass * PIXEL_GRID_Y) / total)
        dx, dy = PIXEL_GRID_X - cx, PIXEL_GRID_Y - cy
        mu02 = float(np.sum(mass * dy * dy) / total)
        if mu02 < 1e-6:
            out[i] = src[i]
            skipped += 1
            continue
        slope = float(np.sum(mass * dx * dy) / total / mu02)
        bounded = float(np.clip(slope, -MAX_SLOPE, MAX_SLOPE))
        if bounded != slope:
            clipped += 1
        if abs(bounded) <= 1e-7:
            out[i] = src[i]
            skipped += 1
            continue
        xsrc = PIXEL_GRID_X + bounded * (PIXEL_GRID_Y - cy)
        corrected = map_coordinates(img, [PIXEL_GRID_Y, xsrc], order=1,
                                    mode='constant', cval=0.0, prefilter=False)
        out[i] = np.clip(corrected, 0, 255).reshape(784).astype(np.float32)
        changed += 1
    return out, {'total': int(len(src)), 'changed': changed,
                 'skipped': skipped, 'clamped': clipped,
                 'threshold': THRESHOLD, 'max_abs_slope': MAX_SLOPE,
                 'interpolation': 'bilinear', 'border': 'constant_zero'}


def test_deskew():
    a = np.zeros((3, 28, 28), dtype=np.float32)
    a[0, 6:23, 13:16] = 255.0
    for y in range(6, 23):
        x = int(round(12 + 0.3 * (y - 14)))
        a[1, y, x:x+3] = 255.0
    transformed, info = deskew(a.reshape(-1, 784))
    assert transformed.shape == (3, 784)
    assert np.all(np.isfinite(transformed)) and 0 <= transformed.min() <= transformed.max() <= 255
    assert np.all(transformed[2] == 0)
    assert info['skipped'] >= 1
    def slope_of(im):
        m = im.reshape(28, 28).astype(np.float64)
        t = m.sum()
        cx = (m * PIXEL_GRID_X).sum() / t
        cy = (m * PIXEL_GRID_Y).sum() / t
        return ((m * (PIXEL_GRID_X - cx) * (PIXEL_GRID_Y - cy)).sum()
                / (m * (PIXEL_GRID_Y - cy) ** 2).sum())
    assert abs(slope_of(transformed[1])) < abs(slope_of(a[1])), 'Synthetic skew not reduced'
    print('DESKEW SELF-TEST PASSED: shape, blank, shear correction, range', flush=True)


def process(batch, kind, com):
    if kind == 'com_control':
        return com.transform(batch), {'kind': 'com_only'}
    if kind == 'deskew_then_com':
        prepared, diagnostics = deskew(batch)
        return com.transform(prepared), diagnostics
    if kind == 'com_then_deskew':
        prepared = com.transform(batch)
        return deskew(prepared)
    raise ValueError(kind)


def main():
    test_augmentation()
    test_deskew()
    xtr, ytr, xva, yva, dataset_sha = fixed_split()
    xhm, yhm, writers, sample_ids = read_handmade()
    if (len(ytr), len(yva), len(yhm)) != (76800, 19200, 2560):
        raise ValueError('Unexpected split sizes')
    print('Training=76800; Official Validation=19200; Hand-made=2560', flush=True)
    idx = sample_balanced(ytr, fraction=AUG_FRACTION, seed=SEED)
    sized, size_stats = size_jitter(xtr, idx, seed=SEED)
    aug, stroke_stats = stroke_jitter(sized, strength=STROKE_STRENGTH)
    train_raw = np.concatenate((xtr, aug), axis=0)
    train_y = np.concatenate((ytr, ytr[idx]), axis=0)
    assert len(train_y) == 99840
    print(f'EXACT EXP-009 augmentation: {len(idx)} extra official Training only, '
          f'strength={STROKE_STRENGTH}', flush=True)

    com = BatchedCoM()
    candidate_rows = []
    writer_rows = []
    class_rows = []
    detail = {}
    preds = {}
    OUTPUT.mkdir(parents=True, exist_ok=True)
    for kind in ('com_control', 'deskew_then_com', 'com_then_deskew'):
        print(f'\n[{kind}] Preparing training and evaluation...', flush=True)
        t = time.perf_counter()
        train, train_diag = process(train_raw, kind, com)
        val, val_diag = process(xva, kind, com)
        hand, hand_diag = process(xhm, kind, com)
        prep_seconds = time.perf_counter() - t
        model = knn_model()
        t = time.perf_counter()
        model.fit(train, train_y)
        fit_seconds = time.perf_counter() - t
        t = time.perf_counter()
        pred_val = np.asarray(model.predict(val), dtype=np.int64)
        val_seconds = time.perf_counter() - t
        t = time.perf_counter()
        pred_hand = np.asarray(model.predict(hand), dtype=np.int64)
        hand_seconds = time.perf_counter() - t
        off, hm = scores(yva, pred_val), scores(yhm, pred_hand)
        correct = (int((yva == pred_val).sum()), int((yhm == pred_hand).sum()))
        if kind == 'com_control':
            if correct != EXPECTED_CONTROL_CORRECT:
                raise RuntimeError(f'EXP-009 reference mismatch: {correct} != {EXPECTED_CONTROL_CORRECT}')
            print('EXP-009 BEST REPRODUCED EXACTLY', flush=True)
        eligible = off['accuracy'] >= OFFICIAL_GUARDRAIL
        row = {
            'variant': kind, 'train_n': int(len(train_y)), 'augmentation_n': int(len(idx)),
            'official_correct': correct[0], 'official_accuracy': off['accuracy'],
            'official_macro_f1': off['macro_f1'], 'handmade_correct': correct[1],
            'handmade_accuracy': hm['accuracy'], 'handmade_macro_f1': hm['macro_f1'],
            'official_guardrail_pass': int(eligible),
            'prep_seconds': prep_seconds, 'fit_seconds': fit_seconds,
            'official_predict_seconds': val_seconds,
            'handmade_predict_seconds': hand_seconds,
        }
        candidate_rows.append(row)
        detail[kind] = {'official': off, 'handmade': hm,
                        'deskew_diagnostics': {'training': train_diag,
                                               'official': val_diag, 'handmade': hand_diag}}
        preds[kind] = (pred_val, pred_hand)
        for writer in (41, 42, 43, 44):
            mask = writers == writer
            ss = scores(yhm[mask], pred_hand[mask])
            writer_rows.append({'variant': kind, 'writer': writer,
                                'n': int(mask.sum()), 'accuracy': ss['accuracy'],
                                'macro_f1': ss['macro_f1']})
        for label_name in LABELS:
            p = hm['per_class'][label_name]
            class_rows.append({'variant': kind, 'class': label_name,
                               'support': p['support'], 'precision': p['precision'],
                               'recall': p['recall'], 'f1': p['f1']})
        save_csv(OUTPUT / 'candidate_results.csv', candidate_rows)
        save_csv(OUTPUT / 'by_writer.csv', writer_rows)
        save_csv(OUTPUT / 'per_class_handmade.csv', class_rows)
        save_cm(OUTPUT / f'official_cm_{kind}.csv', off['confusion_matrix'])
        save_cm(OUTPUT / f'handmade_cm_{kind}.csv', hm['confusion_matrix'])
        save_csv(OUTPUT / f'handmade_predictions_{kind}.csv', [
            {'writer': int(writers[i]), 'index': int(sample_ids[i]),
             'true': LABELS[int(yhm[i])], 'predicted': LABELS[int(pred_hand[i])],
             'correct': int(yhm[i] == pred_hand[i])}
            for i in range(len(yhm))
        ])
        (OUTPUT / f'diagnostics_{kind}.json').write_text(
            json.dumps(detail[kind]['deskew_diagnostics'], ensure_ascii=False, indent=2),
            encoding='utf-8')
        print(f"{kind:<20} Official={off['accuracy']:.6f} F1={off['macro_f1']:.6f} | "
              f"Hand-made={hm['accuracy']:.6f} F1={hm['macro_f1']:.6f} "
              f"| guardrail={'PASS' if eligible else 'FAIL'}", flush=True)
        del model, train, val, hand

    paired = {
        name: {
            'official': paired_stats(yva, preds['com_control'][0], preds[name][0]),
            'handmade': paired_stats(yhm, preds['com_control'][1], preds[name][1]),
        }
        for name in ('deskew_then_com', 'com_then_deskew')
    }
    valid = [row for row in candidate_rows if row['official_guardrail_pass']]
    winner = max(valid, key=lambda r: (r['handmade_macro_f1'], r['handmade_accuracy'],
                                        r['official_macro_f1'])) if valid else None
    metrics = {
        'experiment': 'EXP-011 KNN Normal moment-based deskew comparison',
        'official_dataset_sha256': dataset_sha,
        'train_n': int(len(ytr)), 'official_validation_n': int(len(yva)),
        'handmade_n': int(len(yhm)),
        'knn': {'n_neighbors': 3, 'weights': 'distance', 'p': 2},
        'augmentation': {'fraction': AUG_FRACTION, 'stroke_strength': STROKE_STRENGTH,
                         'augmented_n': len(idx), 'size_counts': size_stats,
                         'stroke_stats': stroke_stats},
        'deskew': {'threshold': THRESHOLD, 'max_abs_slope': MAX_SLOPE,
                   'interpolation': 'bilinear', 'border': 'zero'},
        'official_guardrail_accuracy': OFFICIAL_GUARDRAIL,
        'candidates': candidate_rows, 'selected_development_candidate': winner,
        'paired_vs_control': paired,
        'environment': {'python': platform.python_version(),
                        'numpy': np.__version__, 'sklearn': sklearn.__version__},
        'note': 'Hand-made is repeatedly used development data; not independent final test. '
                'All augmentation uses official training only. No held-out test files loaded.',
    }
    (OUTPUT / 'metrics.json').write_text(
        json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')
    print('\nPAIRED VS EXP-009 BEST:', paired, flush=True)
    print('SELECTED DEVELOPMENT CANDIDATE:', winner['variant'] if winner else 'NONE', flush=True)
    if winner:
        print(f"Official={winner['official_accuracy']:.6f}, Hand-made={winner['handmade_accuracy']:.6f}, "
              f"MacroF1={winner['handmade_macro_f1']:.6f}", flush=True)
    print('Saved:', OUTPUT, flush=True)


if __name__ == '__main__':
    main()


## EXP012_KNN_Normal_HOG

원본: [05_model_optimize/KNN/Normal/scripts/EXP012_KNN_Normal_HOG.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/scripts/EXP012_KNN_Normal_HOG.py)  
Git blob: `bc8ef6102d72ba49e0d9f4dc49ca66cb7adec6b4`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/logs)

EXP-012: KNN Normal pixel vs HOG vs pixel+HOG.

Run from the repository root:
    python 05_model_optimize/KNN/Normal/scripts/EXP012_KNN_Normal_HOG.py

Place next to EXP004_KNN_Normal_CoM.py, EXP006_KNN_Normal_BB_Size_fixed.py,
EXP007_KNN_Normal_Augmentation.py, EXP011_KNN_Normal_Deskew.py.

Official training only (original + balanced 30% size/stroke augmentation).
Evaluation: unchanged official Validation and Hand-made (development only).
No MLP/CNN/held-out official Test data.



In [ ]:
%%archive_source 05_model_optimize/KNN/Normal/scripts/EXP012_KNN_Normal_HOG.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""EXP-012: KNN Normal pixel vs HOG vs pixel+HOG.

Run from the repository root:
    python 05_model_optimize/KNN/Normal/scripts/EXP012_KNN_Normal_HOG.py

Place next to EXP004_KNN_Normal_CoM.py, EXP006_KNN_Normal_BB_Size_fixed.py,
EXP007_KNN_Normal_Augmentation.py, EXP011_KNN_Normal_Deskew.py.

Official training only (original + balanced 30% size/stroke augmentation).
Evaluation: unchanged official Validation and Hand-made (development only).
No MLP/CNN/held-out official Test data.
"""
from __future__ import annotations

import json
import platform
import sys
import time
from pathlib import Path

import numpy as np
import sklearn
from sklearn.neighbors import KNeighborsClassifier

ROOT = Path(__file__).resolve().parents[4]
SCRIPTS = ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'scripts'
sys.path.insert(0, str(ROOT / '03_experiments' / 'model_optimization'))
sys.path.insert(0, str(SCRIPTS))

from common import fixed_split, LABELS, scores
from EXP004_KNN_Normal_CoM import BatchedCoM, read_handmade, paired_stats
from EXP007_KNN_Normal_Augmentation import (
    sample_balanced, size_jitter, stroke_jitter, test_augmentation,
    knn_model, save_csv, save_cm, OFFICIAL_GUARDRAIL, SEED,
)
from EXP011_KNN_Normal_Deskew import deskew, test_deskew

OUTPUT = ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'results' / 'EXP012_hog'
AUGMENTATION_FRACTION = 0.30
STROKE_STRENGTH = 0.44
EXPECTED_EXP011_CONTROL = (18571, 2215)  # 0.96723958*19200; 0.865234375*2560
HOG_CELLS = 4
HOG_CELL_SIDE = 7
HOG_BINS = 9
HOG_FEATURE_DIM = 324
FEATURE_BATCH = 256


def hog_features(raw, batch_size=FEATURE_BATCH):
    """Unsigned 9-bin HOG, 7x7 cells, 2x2 L2-Hys-like normalized blocks.

    28x28 pixels => 4x4 cells => 3x3 overlapping blocks, 324 features.
    Gradient via central differences (numpy.gradient), bilinear vote to
    orientation bins, unsigned angles in [0,180). No label information.
    """
    images = np.asarray(raw, dtype=np.float32)
    if images.ndim != 2 or images.shape[1] != 784:
        raise ValueError(f'Expected N x 784 array; got {images.shape}')
    output = np.empty((len(images), HOG_FEATURE_DIM), dtype=np.float32)
    iy, ix = np.indices((28, 28))
    cell_index = ((iy // HOG_CELL_SIDE) * HOG_CELLS + ix // HOG_CELL_SIDE).reshape(1, 784)
    for start in range(0, len(images), batch_size):
        stop = min(start + batch_size, len(images))
        a = images[start:stop].reshape(-1, 28, 28) / 255.0
        gy, gx = np.gradient(a, axis=(1, 2))
        mag = np.hypot(gx, gy).reshape(-1, 784)
        ori = (np.arctan2(gy, gx) % np.pi).reshape(-1, 784) * (HOG_BINS / np.pi)
        low = np.floor(ori).astype(np.int32) % HOG_BINS
        high = (low + 1) % HOG_BINS
        high_weight = ori - np.floor(ori)
        n = stop - start
        batch_ix = np.arange(n, dtype=np.int32)[:, None]
        hist = np.zeros((n, 16 * HOG_BINS), dtype=np.float32)
        np.add.at(hist, (batch_ix, cell_index * HOG_BINS + low), mag * (1.0 - high_weight))
        np.add.at(hist, (batch_ix, cell_index * HOG_BINS + high), mag * high_weight)
        cells = hist.reshape(n, 4, 4, HOG_BINS)
        blocks = [cells[:, cy:cy+2, cx:cx+2, :].reshape(n, 36)
                  for cy in range(3) for cx in range(3)]
        stacked = np.concatenate(blocks, axis=1).reshape(n, 9, 36)
        norm = np.linalg.norm(stacked, axis=2, keepdims=True)
        normalized = stacked / np.maximum(norm, 1e-6)
        clipped = np.minimum(normalized, 0.2)
        renorm = np.linalg.norm(clipped, axis=2, keepdims=True)
        normalized = clipped / np.maximum(renorm, 1e-6)
        output[start:stop] = normalized.reshape(n, HOG_FEATURE_DIM)
    return output


def unit_rows(x):
    a = np.asarray(x, dtype=np.float32)
    return a / np.maximum(np.linalg.norm(a, axis=1, keepdims=True), 1e-6)


def hybrid_features(pixel, hog, pixel_weight=0.5):
    """Feature-level fusion; pixel and HOG L2-normalized per sample."""
    if pixel_weight <= 0 or pixel_weight >= 1:
        raise ValueError('pixel_weight must lie strictly between zero and one')
    p = unit_rows(np.asarray(pixel, dtype=np.float32) / 255.0)
    h = unit_rows(hog)
    return np.ascontiguousarray(np.concatenate([
        np.sqrt(pixel_weight) * p,
        np.sqrt(1 - pixel_weight) * h,
    ], axis=1), dtype=np.float32)


def test_hog():
    a = np.zeros((3, 28, 28), dtype=np.float32)
    a[0, 5:22, 13:15] = 255
    a[1, 13:15, 5:22] = 255
    a[2] = 0
    h = hog_features(a.reshape(3, 784), batch_size=2)
    assert h.shape == (3, HOG_FEATURE_DIM)
    assert np.isfinite(h).all() and np.all(h >= 0)
    assert np.all(h[2] == 0) and np.linalg.norm(h[0] - h[1]) > 0.1
    mix = hybrid_features(a.reshape(3, 784), h)
    assert mix.shape == (3, 784 + HOG_FEATURE_DIM)
    assert np.isfinite(mix).all()
    print('HOG SELF-TEST PASSED: 324 dims, blank image, orientation sensitivity, hybrid', flush=True)


def evaluate(variant, xtrain, ytrain, xval, yval, xhand, yhand,
             writers, local_indices, control_predictions, candidate_rows,
             writer_rows, class_rows, records):
    print(f'\n[{variant}] Fit and predict...', flush=True)
    t = time.perf_counter()
    if variant == 'pixel_control':
        model = knn_model()  # identical to EXP-011: PixelScaler255 + KNN
    else:
        model = KNeighborsClassifier(n_neighbors=3, weights='distance',
                                     metric='minkowski', p=2, algorithm='brute', n_jobs=-1)
    model.fit(xtrain, ytrain)
    fit_seconds = time.perf_counter() - t
    t = time.perf_counter()
    pv = np.asarray(model.predict(xval), dtype=np.int64)
    val_seconds = time.perf_counter() - t
    t = time.perf_counter()
    ph = np.asarray(model.predict(xhand), dtype=np.int64)
    hand_seconds = time.perf_counter() - t
    off, hm = scores(yval, pv), scores(yhand, ph)
    correct = (int((yval == pv).sum()), int((yhand == ph).sum()))
    if variant == 'pixel_control':
        if correct != EXPECTED_EXP011_CONTROL:
            raise RuntimeError(f'EXP-011 control mismatch: got {correct}, expected {EXPECTED_EXP011_CONTROL}')
        control_predictions['official'], control_predictions['handmade'] = pv.copy(), ph.copy()
        print('EXP-011 BEST REPRODUCED EXACTLY', flush=True)
    eligible = off['accuracy'] >= OFFICIAL_GUARDRAIL
    candidate_rows.append({
        'variant': variant, 'train_n': len(ytrain), 'features': xtrain.shape[1],
        'official_correct': correct[0], 'official_accuracy': off['accuracy'],
        'official_macro_f1': off['macro_f1'], 'handmade_correct': correct[1],
        'handmade_accuracy': hm['accuracy'], 'handmade_macro_f1': hm['macro_f1'],
        'official_guardrail_pass': int(eligible),
        'fit_seconds': fit_seconds, 'official_predict_seconds': val_seconds,
        'handmade_predict_seconds': hand_seconds,
    })
    records[variant] = {'official': off, 'handmade': hm, 'features': xtrain.shape[1],
                        'fit_seconds': fit_seconds, 'official_predict_seconds': val_seconds,
                        'handmade_predict_seconds': hand_seconds}
    for w in (41, 42, 43, 44):
        mask = writers == w
        s = scores(yhand[mask], ph[mask])
        writer_rows.append({'variant': variant, 'writer': w, 'n': int(mask.sum()),
                            'accuracy': s['accuracy'], 'macro_f1': s['macro_f1']})
    for c in LABELS:
        p = hm['per_class'][c]
        class_rows.append({'variant': variant, 'class': c, 'support': p['support'],
                           'precision': p['precision'], 'recall': p['recall'], 'f1': p['f1']})
    save_csv(OUTPUT / 'candidate_results.csv', candidate_rows)
    save_csv(OUTPUT / 'by_writer.csv', writer_rows)
    save_csv(OUTPUT / 'per_class_handmade.csv', class_rows)
    save_cm(OUTPUT / f'official_cm_{variant}.csv', off['confusion_matrix'])
    save_cm(OUTPUT / f'handmade_cm_{variant}.csv', hm['confusion_matrix'])
    save_csv(OUTPUT / f'handmade_predictions_{variant}.csv', [
        {'writer': int(writers[i]), 'index': int(local_indices[i]),
         'true': LABELS[int(yhand[i])], 'predicted': LABELS[int(ph[i])],
         'correct': int(yhand[i] == ph[i])}
        for i in range(len(yhand))
    ])
    if variant != 'pixel_control':
        records[variant]['paired_vs_control'] = {
            'official': paired_stats(yval, control_predictions['official'], pv),
            'handmade': paired_stats(yhand, control_predictions['handmade'], ph),
        }
    (OUTPUT / 'metrics_partial.json').write_text(
        json.dumps(records, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f"{variant:<18} Official={off['accuracy']:.6f} F1={off['macro_f1']:.6f} | "
          f"Hand-made={hm['accuracy']:.6f} F1={hm['macro_f1']:.6f} | "
          f"guardrail={'PASS' if eligible else 'FAIL'} | "
          f"predict_s={val_seconds:.1f}+{hand_seconds:.1f}", flush=True)
    del model


def main():
    test_augmentation()
    test_deskew()
    test_hog()
    xtr, ytr, xva, yva, sha = fixed_split()
    xhm, yhm, writers, local_indices = read_handmade()
    assert (len(ytr), len(yva), len(yhm)) == (76800, 19200, 2560)
    print(f'Training={len(ytr)}, Official Validation={len(yva)}, Hand-made={len(yhm)}', flush=True)
    selected = sample_balanced(ytr, fraction=AUGMENTATION_FRACTION, seed=SEED)
    sized, size_stats = size_jitter(xtr, selected, seed=SEED)
    extra, stroke_stats = stroke_jitter(sized, strength=STROKE_STRENGTH)
    raw_train = np.concatenate((xtr, extra), axis=0)
    train_labels = np.concatenate((ytr, ytr[selected]), axis=0)
    assert len(train_labels) == 99840
    print(f'Augmented official Training only: {len(selected)} extra rows', flush=True)
    com = BatchedCoM()
    t = time.perf_counter()
    xtrain_com = com.transform(raw_train)
    xval_com = com.transform(xva)
    xhand_com = com.transform(xhm)
    xtrain, _ = deskew(xtrain_com)
    xval, _ = deskew(xval_com)
    xhand, _ = deskew(xhand_com)
    print(f'CoM -> Deskew preprocessed all inputs in {time.perf_counter()-t:.1f}s', flush=True)
    del raw_train, xtrain_com, xval_com, xhand_com
    OUTPUT.mkdir(parents=True, exist_ok=True)
    rows, by_writer, per_class, records, control_preds = [], [], [], {}, {}

    evaluate('pixel_control', xtrain, train_labels, xval, yva, xhand, yhm,
             writers, local_indices, control_preds, rows, by_writer, per_class, records)

    t = time.perf_counter()
    train_hog = hog_features(xtrain)
    val_hog = hog_features(xval)
    hand_hog = hog_features(xhand)
    print(f'HOG extraction complete in {time.perf_counter()-t:.1f}s', flush=True)

    evaluate('hog_only', train_hog, train_labels, val_hog, yva, hand_hog, yhm,
             writers, local_indices, control_preds, rows, by_writer, per_class, records)

    train_mix = hybrid_features(xtrain, train_hog, pixel_weight=0.5)
    val_mix = hybrid_features(xval, val_hog, pixel_weight=0.5)
    hand_mix = hybrid_features(xhand, hand_hog, pixel_weight=0.5)
    evaluate('pixel_hog_50_50', train_mix, train_labels, val_mix, yva, hand_mix, yhm,
             writers, local_indices, control_preds, rows, by_writer, per_class, records)

    valid = [r for r in rows if r['official_guardrail_pass']]
    winner = max(valid, key=lambda r: (r['handmade_macro_f1'], r['handmade_accuracy'],
                                      r['official_macro_f1'])) if valid else None
    payload = {
        'experiment': 'EXP-012', 'dataset_sha256': sha,
        'official_train_n': int(len(ytr)), 'augmented_train_n': len(selected),
        'fit_n': len(train_labels), 'official_validation_n': len(yva),
        'handmade_dev_n': len(yhm), 'augmentation': {'fraction': AUGMENTATION_FRACTION,
        'stroke_strength': STROKE_STRENGTH, 'target_long_sides': [16, 18, 22, 24],
        'size_stats': size_stats, 'stroke_stats': stroke_stats},
        'preprocessing': 'CoM then EXP-011 deskew on all splits',
        'knn': {'k': 3, 'weights': 'distance', 'metric': 'euclidean'},
        'hog': {'cell_px': 7, 'n_cells': 4, 'bins': 9,
                'block_cells': 2, 'block_stride_cells': 1, 'feature_dim': HOG_FEATURE_DIM},
        'hybrid': {'pixel_weight': 0.5, 'per_sample_l2_normalized': True},
        'official_guardrail': OFFICIAL_GUARDRAIL,
        'candidates': rows, 'selected_development_candidate': winner,
        'full_results': records,
        'environment': {'python': platform.python_version(), 'numpy': np.__version__,
                        'sklearn': sklearn.__version__},
        'note': 'Hand-made is repeatedly used for model development, NOT independent final testing. '
                'Only official Training used to create extra samples. No official Test used.',
    }
    (OUTPUT / 'metrics.json').write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    print('\nSELECTED DEVELOPMENT CANDIDATE:', winner['variant'] if winner else 'NONE', flush=True)
    print('Saved:', OUTPUT, flush=True)


if __name__ == '__main__':
    main()


## EXP013_KNN_Normal_HOG_Mix_Ratio

원본: [05_model_optimize/KNN/Normal/scripts/EXP013_KNN_Normal_HOG_Mix_Ratio.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/scripts/EXP013_KNN_Normal_HOG_Mix_Ratio.py)  
Git blob: `6262bddbd2c8a459bad6ee2d9bca3b64edc2a103`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/logs)

EXP-013: Pixel/HOG feature weighting for KNN Normal.

Run in repository root:
python 05_model_optimize/KNN/Normal/scripts/EXP013_KNN_Normal_HOG_Mix_Ratio.py

Requires existing EXP004, EXP007, EXP011, EXP012 scripts in the same directory.
Original official Training only; no official Test or Hand-made training.



In [ ]:
%%archive_source 05_model_optimize/KNN/Normal/scripts/EXP013_KNN_Normal_HOG_Mix_Ratio.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""EXP-013: Pixel/HOG feature weighting for KNN Normal.

Run in repository root:
python 05_model_optimize/KNN/Normal/scripts/EXP013_KNN_Normal_HOG_Mix_Ratio.py

Requires existing EXP004, EXP007, EXP011, EXP012 scripts in the same directory.
Original official Training only; no official Test or Hand-made training.
"""
from __future__ import annotations

import json
import platform
import sys
import time
from pathlib import Path

import numpy as np
import sklearn
from sklearn.metrics import confusion_matrix
from sklearn.neighbors import KNeighborsClassifier

ROOT = Path(__file__).resolve().parents[4]
SCRIPTS = ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'scripts'
sys.path.insert(0, str(ROOT / '03_experiments' / 'model_optimization'))
sys.path.insert(0, str(SCRIPTS))

from common import fixed_split, LABELS, scores
from EXP004_KNN_Normal_CoM import BatchedCoM, read_handmade, paired_stats
from EXP007_KNN_Normal_Augmentation import (
    sample_balanced, size_jitter, stroke_jitter, test_augmentation,
    save_csv, save_cm, OFFICIAL_GUARDRAIL, SEED,
)
from EXP011_KNN_Normal_Deskew import deskew, test_deskew
from EXP012_KNN_Normal_HOG import hog_features, hybrid_features, test_hog, unit_rows

OUT = ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'results' / 'EXP013_hog_mix_ratio'
WEIGHTS = (0.0, 0.2, 0.35, 0.5, 0.65)
FRACTION = 0.30
STROKE = 0.44
REFERENCE_HOG = (18667, 2341)  # verified from EXP-012 (official, handmade correct)
REFERENCE_MIX50 = (18854, 2341)


def features(pixel, hog, weight):
    if weight == 0.0:
        return hog
    return hybrid_features(pixel, hog, pixel_weight=weight)


def evaluate(name, weight, xtr, ytr, xva, yva, xhm, yhm, writers, indices,
             records, rows, by_writer, per_class, baseline_preds):
    print(f'\n[{name}] KNN fit / predict...', flush=True)
    model = KNeighborsClassifier(n_neighbors=3, weights='distance',
                                 metric='euclidean', algorithm='brute', n_jobs=-1)
    t = time.perf_counter()
    model.fit(xtr, ytr)
    fit_s = time.perf_counter() - t
    t = time.perf_counter()
    pv = np.asarray(model.predict(xva), dtype=np.int64)
    val_s = time.perf_counter() - t
    t = time.perf_counter()
    ph = np.asarray(model.predict(xhm), dtype=np.int64)
    hm_s = time.perf_counter() - t
    off, handmade = scores(yva, pv), scores(yhm, ph)
    correct = (int(np.sum(pv == yva)), int(np.sum(ph == yhm)))
    if weight == 0.0:
        if correct != REFERENCE_HOG:
            raise RuntimeError(f'EXP-012 HOG reference mismatch: expected {REFERENCE_HOG}, got {correct}')
        baseline_preds['official'], baseline_preds['handmade'] = pv.copy(), ph.copy()
        print('EXP-012 HOG baseline reproduced exactly', flush=True)
    if weight == 0.5:
        if correct != REFERENCE_MIX50:
            raise RuntimeError(f'EXP-012 50:50 reference mismatch: expected {REFERENCE_MIX50}, got {correct}')
        print('EXP-012 50:50 reference reproduced exactly', flush=True)
    row = {
        'variant': name, 'pixel_weight': weight, 'hog_weight': 1.0 - weight,
        'train_n': len(ytr), 'n_features': xtr.shape[1],
        'official_correct': correct[0], 'official_accuracy': off['accuracy'],
        'official_macro_f1': off['macro_f1'],
        'handmade_correct': correct[1], 'handmade_accuracy': handmade['accuracy'],
        'handmade_macro_f1': handmade['macro_f1'],
        'official_guardrail_pass': int(off['accuracy'] >= OFFICIAL_GUARDRAIL),
        'fit_seconds': fit_s, 'official_predict_seconds': val_s,
        'handmade_predict_seconds': hm_s,
    }
    rows.append(row)
    for w in (41, 42, 43, 44):
        mask = writers == w
        sc = scores(yhm[mask], ph[mask])
        by_writer.append({'variant': name, 'writer': w, 'n': int(np.sum(mask)),
                          'accuracy': sc['accuracy'], 'macro_f1': sc['macro_f1']})
    for c in LABELS:
        s = handmade['per_class'][c]
        per_class.append({'variant': name, 'class': c, 'support': s['support'],
                          'precision': s['precision'], 'recall': s['recall'], 'f1': s['f1']})
    records[name] = {'row': row, 'official': off, 'handmade': handmade}
    if weight != 0.0:
        records[name]['paired_vs_hog'] = {
            'official': paired_stats(yva, baseline_preds['official'], pv),
            'handmade': paired_stats(yhm, baseline_preds['handmade'], ph),
        }
    save_csv(OUT / 'candidate_results.csv', rows)
    save_csv(OUT / 'by_writer.csv', by_writer)
    save_csv(OUT / 'per_class_handmade.csv', per_class)
    save_cm(OUT / f'official_cm_{name}.csv', off['confusion_matrix'])
    save_cm(OUT / f'handmade_cm_{name}.csv', handmade['confusion_matrix'])
    save_csv(OUT / f'handmade_predictions_{name}.csv', [
        {'writer': int(writers[i]), 'index': int(indices[i]),
         'true': LABELS[int(yhm[i])], 'predicted': LABELS[int(ph[i])],
         'correct': int(yhm[i] == ph[i])} for i in range(len(yhm))
    ])
    (OUT / 'metrics_partial.json').write_text(
        json.dumps(records, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f"{name:<19} Official={off['accuracy']:.6f} F1={off['macro_f1']:.6f} | "
          f"Hand-made={handmade['accuracy']:.6f} F1={handmade['macro_f1']:.6f} | "
          f"guardrail={'PASS' if row['official_guardrail_pass'] else 'FAIL'} | "
          f'predict_s={val_s:.1f}+{hm_s:.1f}', flush=True)
    del model


def main():
    test_augmentation()
    test_deskew()
    test_hog()
    assert np.isclose(sum((0.35, 0.65)), 1.0)
    xtr, ytr, xva, yva, sha = fixed_split()
    xhm, yhm, writers, indices = read_handmade()
    assert (len(ytr), len(yva), len(yhm)) == (76800, 19200, 2560)
    selected = sample_balanced(ytr, fraction=FRACTION, seed=SEED)
    sized, size_stats = size_jitter(xtr, selected, seed=SEED)
    extra, stroke_stats = stroke_jitter(sized, strength=STROKE)
    raw_train = np.concatenate((xtr, extra), axis=0)
    train_labels = np.concatenate((ytr, ytr[selected]), axis=0)
    assert len(train_labels) == 99840
    print(f'Training={len(ytr)}, Official Validation={len(yva)}, Hand-made={len(yhm)}', flush=True)
    print(f'Augmented official Training only: {len(selected)} rows', flush=True)
    t = time.perf_counter()
    com = BatchedCoM()
    xtrain, _ = deskew(com.transform(raw_train))
    xval, _ = deskew(com.transform(xva))
    xhand, _ = deskew(com.transform(xhm))
    del raw_train, extra, sized
    print(f'CoM -> Deskew finished in {time.perf_counter() - t:.1f}s', flush=True)
    t = time.perf_counter()
    htr = hog_features(xtrain)
    hva = hog_features(xval)
    hhm = hog_features(xhand)
    print(f'HOG extracted once in {time.perf_counter() - t:.1f}s', flush=True)
    OUT.mkdir(parents=True, exist_ok=True)
    rows, by_writer, per_class, records, baseline_preds = [], [], [], {}, {}
    for weight in WEIGHTS:
        name = 'hog_only' if weight == 0.0 else f'pixel_{int(round(weight * 100)):02d}_hog_{int(round((1-weight)*100)):02d}'
        train_features = features(xtrain, htr, weight)
        val_features = features(xval, hva, weight)
        hand_features = features(xhand, hhm, weight)
        evaluate(name, weight, train_features, train_labels, val_features, yva,
                 hand_features, yhm, writers, indices, records, rows,
                 by_writer, per_class, baseline_preds)
        del train_features, val_features, hand_features
    eligible = [r for r in rows if r['official_guardrail_pass']]
    winner = max(eligible, key=lambda r: (r['handmade_macro_f1'],
                                         r['handmade_accuracy'],
                                         r['official_accuracy'])) if eligible else None
    payload = {
        'experiment': 'EXP-013', 'dataset_sha256': sha,
        'official_training_n': len(ytr), 'augmented_training_n': len(selected),
        'train_n_total': len(train_labels), 'validation_n': len(yva),
        'handmade_dev_n': len(yhm),
        'augmentation': {'fraction': FRACTION, 'stroke_strength': STROKE,
                         'size_stats': size_stats, 'stroke_stats': stroke_stats},
        'preprocessing': 'CoM -> Deskew',
        'knn': {'n_neighbors': 3, 'weights': 'distance', 'metric': 'euclidean'},
        'hog_dim': htr.shape[1], 'pixel_dim': xtrain.shape[1],
        'hybrid': 'Per-image L2 normalization of pixel and HOG, concatenated after sqrt-weighting',
        'weights_evaluated': list(WEIGHTS), 'candidate_results': rows,
        'selected_development_candidate': winner, 'results': records,
        'environment': {'python': platform.python_version(), 'numpy': np.__version__,
                        'sklearn': sklearn.__version__},
        'note': 'Hand-made has been reused for model selection; this is development validation, not a final unseen test. Official Test not used.',
    }
    (OUT / 'metrics.json').write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')
    print('\nSELECTED DEVELOPMENT CANDIDATE:', winner['variant'] if winner else 'NONE', flush=True)
    print('Saved:', OUT, flush=True)


if __name__ == '__main__':
    main()


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")